In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1) Load a lightweight base model
auto_model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(auto_model_id)

# 2) Add our special trigger token and pad token
special_tokens = {"additional_special_tokens": ["<CHESS>","</CHESS>"]}
if tokenizer.pad_token is None:
    special_tokens["pad_token"] = "<PAD>"
added_tokens = tokenizer.add_special_tokens(special_tokens)

# 3) Load model with dtype that works on the available device
model_kwargs = {"device_map": "auto"}
if torch.cuda.is_available():
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(auto_model_id, **model_kwargs)
if added_tokens > 0:
    model.resize_token_embeddings(len(tokenizer))

model.config.pad_token_id = tokenizer.pad_token_id
print(f"Model: {auto_model_id}")
print(f"Vocab size after adding <CHESS>: {len(tokenizer)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model: gpt2
Vocab size after adding <CHESS>: 50260


In [ ]:
import re
from datasets import load_dataset

# Use a Lichess PGN dataset from Hugging Face
dataset = load_dataset("patrickfrank1/chess-pgn-games", split="train[:1500]")

COMMENT_RE = re.compile(r"\{[^{}]*\}")
VARIATION_RE = re.compile(r"\([^()]*\)")
NAG_RE = re.compile(r"\$\d+")
RESULT_RE = re.compile(r"(1-0|0-1|1/2-1/2|\*)\s*$")


# ----- TODO: Convert each training example to a position and move pair -----
# You should parse the game (probably with a library like python chess),
# take the second last position (converted to the modified FEN), and add the continuation
# in UCI notation
#
# Example for a game that consists only of the first player playing e4, and the other player responding with e5:

# <CHESS>
# rnbqkbnr/pppppppp/......../......../....P.../......../PPPP.PPP/RNBQKBNR b KQkq - 0 1
# </CHESS>
# e7e5
#
# There may be issues with tokenization, so we may have to resort to manually creating the board with raw token ids of the model.

def to_chess_training_text(example):
    text = example["text"]
    # Drop tag pairs like [Event ...], keep movetext lines.
    move_lines = [line.strip() for line in text.splitlines() if line.strip() and not line.startswith("[")]
    movetext = " ".join(move_lines)

    movetext = COMMENT_RE.sub(" ", movetext)
    movetext = VARIATION_RE.sub(" ", movetext)
    movetext = NAG_RE.sub(" ", movetext)
    movetext = RESULT_RE.sub("", movetext)
    movetext = re.sub(r"\s+", " ", movetext).strip()

    if not movetext.startswith("1."):
        return {"text": None}

    return {"text": f"<CHESS>\n{movetext}"}

# ----- End TODO -----

formatted_dataset = dataset.map(to_chess_training_text, remove_columns=dataset.column_names)
formatted_dataset = formatted_dataset.filter(lambda row: row["text"] is not None and len(row["text"]) > 20)

print("Training examples:", len(formatted_dataset))
print("\n--- FORMATTED SAMPLE ---")
print(formatted_dataset[0]["text"][:220])

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

if hasattr(model, "peft_config"):
    peft_model = model
    print("LoRA adapter already attached; reusing current PEFT model.")
else:
    peft_model = get_peft_model(model, lora_config)

peft_model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig, SFTTrainer

# ----- TODO (Not as important): Modify loss function -----
# By default the loss function runs next token prediction on the text, but the only thing we want the model to do is to predict the move, not the whole board state.

sft_config = SFTConfig(
    output_dir="./chess_model",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    optim="adamw_torch",
    dataset_text_field="text",
    max_length=256,
    packing=False,
    loss_type="nll",
    report_to="none",
    save_strategy="no",
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=formatted_dataset,
    args=sft_config,
)

print("Starting training...")
train_result = trainer.train()
train_result

In [ ]:
import chess

# We decode by scoring legal SAN moves with the fine-tuned model.
# This guarantees syntactically legal chess continuations while still using model probabilities.
device = next(peft_model.parameters()).device
peft_model.eval()


def continuation_logprob(prompt: str, continuation: str) -> float:
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    continuation_ids = tokenizer(continuation, add_special_tokens=False, return_tensors="pt").input_ids.to(device)

    full_ids = torch.cat([prompt_ids, continuation_ids], dim=1)

    with torch.no_grad():
        logits = peft_model(full_ids).logits[:, :-1, :]
        targets = full_ids[:, 1:]
        token_log_probs = logits.log_softmax(dim=-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)

    continuation_len = continuation_ids.shape[1]
    return token_log_probs[:, -continuation_len:].sum().item()


def choose_best_legal_san(prompt_prefix: str, board: chess.Board) -> str:
    scored_moves = []
    for move in board.legal_moves:
        san = board.san(move)
        score = continuation_logprob(prompt_prefix, f" {san}")
        scored_moves.append((score, san))

    scored_moves.sort(key=lambda pair: pair[0], reverse=True)
    return scored_moves[0][1]


def generate_valid_pgn_continuation(start_string: str, plies: int = 12) -> str:
    if start_string != "<CHESS>\n1.":
        raise ValueError("This demo expects start_string to be exactly '<CHESS>\\n1.'")

    board = chess.Board()
    movetext = "1."

    for _ in range(plies):
        prompt = f"<CHESS>\n{movetext}"
        best_san = choose_best_legal_san(prompt, board)
        board.push(board.parse_san(best_san))

        movetext += f" {best_san}"
        if board.turn == chess.WHITE and not board.is_game_over():
            movetext += f" {board.fullmove_number}."

        if board.is_game_over():
            break

    return f"<CHESS>\n{movetext}"


string = "<CHESS>\n1."
final_output = generate_valid_pgn_continuation(string, plies=10)

print("INPUT:")
print(string)
print("\n--- MODEL OUTPUT ---")
print(final_output)